# Is the positional penalty caused by the positional embedding itself?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgrfhyL/audio_model_initial_testing/blob/main/Colab_PositionalEmbedding.ipynb)

Moving a short utterance from 5 s to 25 s inside Whisper's 30 s window raises corpus WER sharply
when timestamp tokens are enabled. That could come from several places: the convolutional front
end, attention over 20 s of leading silence, the `max_initial_timestamp` decoding rule, or the
**positional embedding** the encoder adds to every frame.

This notebook separates them by editing the positional embedding directly while leaving the audio
untouched.

| | audio at | positional embedding | first timestamp | asks |
|---|---|---|---|---|
| **P0** | 5 s | unchanged | capped (stock 1.0) | baseline |
| **P1** | 25 s | unchanged | capped | the effect being explained |
| **P2** | 25 s | displaced −20 s | capped | does *relabelling the frames as 5 s* recover P0? |
| **P3** | 25 s | displaced +20 s | capped | does *any* relabelling help, or only the correct one? |
| **P4** | 25 s | unchanged | **uncapped** (`None`) | how much is the decoding rule rather than the model? |

P0–P3 run with timestamps on **and** off; P4 exists only with timestamps on, since without
timestamp tokens there is no initial timestamp to cap. That is 9 arms × 1000 utterances × 5 models
= **45 000 decodes**, ~2 h on a T4.

The audio is byte-identical in every condition. Only the embedding added to it, or the decoding
rule applied to it, changes.

## How the positional embedding is displaced

In `whisper/model.py`, `AudioEncoder.forward` adds a fixed sinusoidal embedding **after** the
convolutions:

```python
x = F.gelu(self.conv1(x))
x = F.gelu(self.conv2(x))          # stride 2: 3000 mel frames -> 1500 encoder frames
x = x.permute(0, 2, 1)
x = (x + self.positional_embedding).to(x.dtype)
```

`positional_embedding` is a `register_buffer` of shape `(1500, n_state)` — fixed sinusoids, not a
learned parameter — so it can be overwritten freely. 1500 frames over 30 s means **50 frames per
second**, so a displacement of *N* seconds is `N * 50` rows.

### Why a cyclic roll, and not extrapolated sinusoids

The obvious implementation is to rebuild the sinusoids at shifted positions,
`sinusoids(arange(1500) - 1000)`. For the *audio* frames that is identical to a roll. But it gives
the 1250 leading silence frames **negative positions**, which the model never saw in training, and
that is catastrophic — measured on 60 clips with the audio at 25 s and timestamps off:

| positional embedding | corpus WER | runaway outputs |
|---|---:|---:|
| unchanged | 0.0586 | 0 / 60 |
| −20 s, extrapolated sinusoids | **6.4163** | **18 / 60** |
| −20 s, cyclic roll | 0.0753 | 0 / 60 |

Extrapolation drives previously-perfect files into repetition loops — the same failure mode the
experiment is studying, induced by out-of-distribution input rather than by position. A cyclic
roll assigns every frame a genuine training-time position and avoids that entirely, so it is what
this notebook uses.

One consequence to state plainly: with a roll, **P3 wraps**. Audio at 25 s displaced by +20 s
would be 45 s, which does not exist in a 30 s window, so it wraps to `(1250 + 1000) mod 1500 = 750`
— the model is told the audio sits at **15 s**. P3 is therefore "relabelled incorrectly but
in-range", which is exactly the control P2 needs.

## 1. Environment

Same base settings as the other sweeps: Colab GPU, fp16 on CUDA, greedy, batch 8.

In [ ]:
!pip -q install openai-whisper jiwer soundfile

import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Provenance

Same record as the delta sweep: package versions, SHA-256 of the `whisper` source files that
define the decode path, each checkpoint verified against the digest embedded in its download URL,
device and precision, verbatim decoding options, and digests over both the audio arrays and the
normalized references.

**Licensing.** TIMIT is LDC93S1 — licensed, not redistributable. No transcript text or audio ever
reaches this notebook's printed output, because saving a Colab notebook back to GitHub commits its
outputs. Hypotheses go to a Drive-only file; the git-safe file carries paths and numbers only, and
is asserted to be so before writing.

In [ ]:
import hashlib, json, os, platform

DRIVE_ROOT  = "/content/drive/MyDrive/NAACL"
RESULTS_CSV = os.path.join(DRIVE_ROOT, "pe_results_full.csv")      # has text -> Drive only
SAFE_CSV    = os.path.join(DRIVE_ROOT, "pe_per_condition.csv")     # numbers only -> git-safe
PROV_JSON   = os.path.join(DRIVE_ROOT, "pe_provenance.json")
LOCAL_AUDIO = "/content/corpus"
REPO        = "AgrfhyL/audio_model_initial_testing"

def sha256_file(path, buf=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(buf), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

import numpy, soundfile, jiwer, whisper

_wdir = os.path.dirname(whisper.__file__)
CODE_DIGESTS = {f: sha256_file(os.path.join(_wdir, f))[:16]
                for f in ("audio.py", "decoding.py", "model.py")}
try:
    import urllib.request
    with urllib.request.urlopen(
            f"https://api.github.com/repos/{REPO}/commits/main", timeout=10) as r:
        REPO_COMMIT = json.load(r)["sha"]
except Exception as e:
    REPO_COMMIT = f"unavailable ({type(e).__name__})"

PROVENANCE = {
    "experiment": "positional-embedding displacement vs. utterance position",
    "conditions": "P0 5s | P1 25s | P2 25s PE-20s | P3 25s PE+20s | P4 25s uncapped first ts",
    "pe_method": "cyclic roll of AudioEncoder.positional_embedding (50 frames/s)",
    "packages": {
        "python": platform.python_version(), "openai-whisper": whisper.__version__,
        "torch": torch.__version__, "numpy": numpy.__version__,
        "soundfile": soundfile.__version__, "jiwer": getattr(jiwer, "__version__", "n/a"),
        "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    },
    "code_version": {"whisper_source_sha256_16": CODE_DIGESTS, "repo": REPO,
                     "repo_commit": REPO_COMMIT},
    "device": {"name": torch.cuda.get_device_name(0),
               "capability": ".".join(map(str, torch.cuda.get_device_capability(0))),
               "precision": "fp16 (encoder/decoder); mel computed fp32 on CPU"},
}
print(json.dumps(PROVENANCE, indent=1))

## 3. Corpus: all 1000 clips

The full seed-0 draw from `corpus_digests.json` — the same corpus every other notebook here
scores, so results join by `path`. Each clip is copied to local disk (Drive's FUSE mount is far
too slow to read repeatedly) and checked against its frozen `sha256_audio`, a digest over the
decoded float32 samples.

In [ ]:
import collections, glob, shutil, time
import numpy as np
import soundfile as sf

cands = [d for d in glob.glob(os.path.join(DRIVE_ROOT, "timit", "**", "TEST"), recursive=True)
         if os.path.isdir(os.path.join(d, "DR1"))]
assert cands, f"no TIMIT TEST/DR1 found under {DRIVE_ROOT}/timit"
TIMIT_TEST = sorted(cands, key=len)[0]

if not os.path.exists("corpus_digests.json"):
    !wget -q https://raw.githubusercontent.com/AgrfhyL/audio_model_initial_testing/main/corpus_digests.json
DIG = json.load(open("corpus_digests.json"))

def sha256_audio(a):
    return hashlib.sha256(np.ascontiguousarray(a, dtype=np.float32).tobytes()).hexdigest()

def load_reference(wav_path):
    with open(wav_path[:-4] + ".TXT") as f:
        return f.read().strip().split(None, 2)[2]

FILES, AUDIO, REFTEXT, bad = [], {}, {}, []
t0 = time.time()
for r in DIG["files"]:
    src = os.path.join(TIMIT_TEST, r["path"]); dst = os.path.join(LOCAL_AUDIO, r["path"])
    if not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy2(src, dst)
    a, sr = sf.read(dst, dtype="float32")
    if sr != DIG["sample_rate"] or len(a) != r["samples"] or sha256_audio(a) != r["sha256_audio"]:
        bad.append(r["path"]); continue
    AUDIO[r["path"]] = a; REFTEXT[r["path"]] = load_reference(src); FILES.append(r)

assert not bad, f"{len(bad)} clips fail their frozen digest, e.g. {bad[:3]}"
assert len(FILES) == 1000
spk = collections.Counter(r["speaker"] for r in FILES)
print(f"verified {len(FILES)} clips in {time.time()-t0:.0f}s | {len(spk)} speakers "
      f"({min(spk.values())}-{max(spk.values())} each) | "
      f"{sum(r['sec'] for r in FILES)/60:.1f} min audio")

## 4. Conditions, positional-embedding control, and the mel gate

`rolled_pe` is a context manager so the buffer is always restored — `load_model` caches, and a
leaked displacement would silently contaminate every later decode. Restoration is asserted, not
assumed.

The mel gate is the control the comparison rests on: 5 s and 25 s must be exact integer-frame
shifts of each other, checked at both filterbank sizes since `large-v3` uses 128 mel bins where
the smaller models use 80.

In [ ]:
import contextlib
from whisper.audio import log_mel_spectrogram, N_SAMPLES, SAMPLE_RATE, HOP_LENGTH
from whisper.normalizers import EnglishTextNormalizer

MODELS   = ["tiny", "base", "small", "medium", "large-v3"]
BATCH    = 8                      # divides 1000 exactly; fits large-v3 on a T4
PE_FPS   = 50                     # 1500 encoder frames / 30 s

# name, offset_s, pe_shift_s, max_initial_timestamp, timestamp arms
CONDITIONS = [
    ("P0",  5, None, 1.0,  (True, False)),
    ("P1", 25, None, 1.0,  (True, False)),
    ("P2", 25, -20,  1.0,  (True, False)),
    ("P3", 25, +20,  1.0,  (True, False)),
    ("P4", 25, None, None, (True,)),        # uncapped: undefined without timestamp tokens
]
OFFSETS_S = sorted({c[1] for c in CONDITIONS})

BASE_OPTS = dict(task="transcribe", language="en", temperature=0.0, beam_size=None,
                 best_of=None, prompt=None, prefix=None, fp16=True,
                 suppress_blank=True, suppress_tokens="-1")


@contextlib.contextmanager
def rolled_pe(model, seconds):
    '''Cyclically roll the encoder positional embedding by `seconds`.

    frame t receives the embedding of frame (t - seconds*PE_FPS) mod n_ctx, so every row is a
    genuine training-time position. Restores the buffer on exit.
    '''
    pe = model.encoder.positional_embedding
    saved = pe.detach().clone()
    try:
        if seconds:
            pe.copy_(torch.roll(saved, shifts=int(-seconds * PE_FPS), dims=0))
        yield
    finally:
        pe.copy_(saved)
        assert torch.equal(pe, saved), "positional embedding was not restored"


def place(audio, off_s):
    buf = np.zeros(N_SAMPLES, dtype=np.float32)
    o = off_s * SAMPLE_RATE
    buf[o:o + len(audio)] = audio
    return buf


def mel_of(audio, off_s, n_mels):
    return log_mel_spectrogram(torch.from_numpy(place(audio, off_s)), n_mels)


normalizer = EnglishTextNormalizer()
REF = {p: normalizer(t) for p, t in REFTEXT.items()}
assert all(v.strip() for v in REF.values())

for s in OFFSETS_S:
    assert (s * SAMPLE_RATE) % HOP_LENGTH == 0
for r in FILES:
    for s in OFFSETS_S:
        assert s * SAMPLE_RATE + r["samples"] <= N_SAMPLES, (r["path"], s)
assert len(FILES) % BATCH == 0, f"{len(FILES)} not divisible by BATCH={BATCH}"

# --- mel gate: 5 s vs 25 s bit-identical, at both filterbank sizes ---
import random as _random
FPS, NF = SAMPLE_RATE // HOP_LENGTH, N_SAMPLES // HOP_LENGTH
gate = {}
for n_mels in (80, 128):
    worst = 0.0
    for r in _random.Random(7).sample(FILES, min(40, len(FILES))):
        a = AUDIO[r["path"]]; nf = int(np.ceil(len(a) / HOP_LENGTH)) + 2
        m5, m25 = mel_of(a, 5, n_mels).numpy(), mel_of(a, 25, n_mels).numpy()
        n = min(nf, NF - 25 * FPS)
        worst = max(worst, float(np.abs(m25[:, 25*FPS:25*FPS+n] - m5[:, 5*FPS:5*FPS+n]).max()))
    gate[n_mels] = worst
    assert worst == 0.0, f"mel gate FAILED at n_mels={n_mels}: {worst:.3e}"

corpus_digest = sha256_bytes("\n".join(
    f"{r['path']}\t{r['sha256_audio']}" for r in FILES).encode())
reference_digest = sha256_bytes("\n".join(
    f"{r['path']}\t{REF[r['path']]}" for r in FILES).encode())
PROVENANCE["decoding"] = {**BASE_OPTS, "greedy": True, "batch": BATCH,
                          "conditions": [(c[0], c[1], c[2], c[3]) for c in CONDITIONS]}
PROVENANCE["corpus"] = {
    "source": "TIMIT (LDC93S1) TEST -- licensed, not redistributable",
    "spec_sha256": sha256_file("corpus_digests.json"), "n_clips": len(FILES),
    "n_speakers": len({r["speaker"] for r in FILES}),
    "corpus_digest_sha256": corpus_digest, "reference_digest_sha256": reference_digest,
}
PROVENANCE["mel_gate"] = {"offsets_compared": OFFSETS_S, "max_abs_dev_by_n_mels": gate}
print("mel gate: 5 s vs 25 s bit-identical at " +
      ", ".join(f"n_mels={k} (|dev| {v:.1e})" for k, v in gate.items()))
print("corpus digest   ", corpus_digest)
print("reference digest", reference_digest)

## 5. The sweep

Ordered model → offset → batch, so the mel for a batch is computed **once** and reused by every
condition sharing that offset (P1–P4 all sit at 25 s). Both timestamp arms then decode off that
same mel, which guarantees they see bit-identical input.

Resume works at (model, condition, arm) granularity: a partially finished arm is re-decoded in
full so every batch is always formed from the same 1000-clip list, keeping batching uniform. Rows
already on disk are not rewritten.

In [ ]:
import csv, gc

FIELDS = ["model", "n_mels", "cond", "offset_s", "pe_shift_s", "max_init_ts", "timestamps",
          "path", "speaker", "text", "avg_logprob", "no_speech_prob"]

def checkpoint_digest(name):
    url = whisper._MODELS[name]
    root = os.path.join(os.getenv("XDG_CACHE_HOME",
                                  os.path.join(os.path.expanduser("~"), ".cache")), "whisper")
    p = os.path.join(root, os.path.basename(url))
    return url.split("/")[-2], (sha256_file(p) if os.path.exists(p) else None)

done = set()
if os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV, newline="") as f:
        for row in csv.DictReader(f):
            done.add((row["model"], row["cond"], row["timestamps"] == "on", row["path"]))
total = sum(len(c[4]) for c in CONDITIONS) * len(FILES) * len(MODELS)
print(f"{len(done)}/{total} cells already done")

os.makedirs(DRIVE_ROOT, exist_ok=True)
ckpt, t_start = {}, time.time()
with open(RESULTS_CSV, "a", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=FIELDS)
    if not done:
        w.writeheader()

    for name in MODELS:
        todo_model = [(c, t) for c in CONDITIONS for t in c[4]
                      if not all((name, c[0], t, r["path"]) in done for r in FILES)]
        if not todo_model:
            print(f"{name}: complete"); continue

        model = whisper.load_model(name, device="cuda")
        exp, act = checkpoint_digest(name)
        assert exp == act, f"{name}: checkpoint digest mismatch"
        n_mels = model.dims.n_mels
        ckpt[name] = {"expected_sha256": exp, "ondisk_sha256": act, "verified": True,
                      "params_M": round(sum(p.numel() for p in model.parameters())/1e6, 1),
                      "n_mels": n_mels, "batch": BATCH}
        t_model = time.time()
        print(f"\n{name}: {ckpt[name]['params_M']}M params, n_mels={n_mels}, "
              f"sha256 {act[:16]}... verified")

        for off_s in OFFSETS_S:
            arms = [(c, t) for (c, t) in todo_model if c[1] == off_s]
            if not arms:
                continue
            t0 = time.time()
            for i in range(0, len(FILES), BATCH):
                batch = FILES[i:i + BATCH]
                mel = torch.stack([mel_of(AUDIO[r["path"]], off_s, n_mels)
                                   for r in batch]).to("cuda")     # computed once per offset
                for (cond, _o, pe_s, mit, _arms), ts_on in arms:
                    if all((name, cond, ts_on, r["path"]) in done for r in batch):
                        continue
                    opts = whisper.DecodingOptions(**BASE_OPTS, without_timestamps=not ts_on,
                                                   max_initial_timestamp=mit)
                    with rolled_pe(model, pe_s):
                        res = whisper.decode(model, mel, opts)
                    for r, d in zip(batch, res):
                        if (name, cond, ts_on, r["path"]) in done:
                            continue
                        w.writerow({"model": name, "n_mels": n_mels, "cond": cond,
                                    "offset_s": off_s, "pe_shift_s": "" if pe_s is None else pe_s,
                                    "max_init_ts": "" if mit is None else mit,
                                    "timestamps": "on" if ts_on else "off",
                                    "path": r["path"], "speaker": r["speaker"], "text": d.text,
                                    "avg_logprob": f"{d.avg_logprob:.6f}",
                                    "no_speech_prob": f"{d.no_speech_prob:.6f}"})
                        done.add((name, cond, ts_on, r["path"]))
                fh.flush()
            print(f"  offset {off_s:2d}s: {len(arms)} arms x {len(FILES)} clips "
                  f"in {time.time()-t0:.0f}s")

        del model; gc.collect(); torch.cuda.empty_cache()
        print(f"  {name} done in {(time.time()-t_model)/60:.1f} min")

PROVENANCE["checkpoints"] = ckpt
print(f"\nsweep complete: {len(done)} cells in {(time.time()-t_start)/60:.1f} min -> {RESULTS_CSV}")

## 6. Corpus WER

Pooled corpus WER — total substitutions + deletions + insertions over total reference words — with
Whisper's `EnglishTextNormalizer` applied to both sides. The **recovery** column is the fraction of
P1's penalty that a condition removes:

```
recovery = (WER(P1) - WER(Px)) / (WER(P1) - WER(P0))
```

1.0 means the condition returns fully to the 5 s baseline; 0.0 means it changes nothing.

In [ ]:
from jiwer import process_words

rows = list(csv.DictReader(open(RESULTS_CSV, newline="")))
print(f"{len(rows)} result rows\n")

HYP = {(r["model"], r["cond"], r["timestamps"], r["path"]): r["text"] for r in rows}
refs = [REF[r["path"]] for r in FILES]

wer = {}
for name in MODELS:
    for cond, _o, _p, _m, arms in CONDITIONS:
        for ts_on in arms:
            ts = "on" if ts_on else "off"
            hyps = [HYP.get((name, cond, ts, r["path"])) for r in FILES]
            assert all(h is not None for h in hyps), f"missing rows for {name}/{cond}/{ts}"
            wer[(name, cond, ts)] = process_words(
                refs, [normalizer(h) for h in hyps]).wer

def runaway(name, cond, ts):
    return sum(1 for r in FILES
               if len(normalizer(HYP[(name, cond, ts, r["path"])]).split())
               > 3 * len(REF[r["path"]].split()))

LABEL = {"P0": "5s baseline", "P1": "25s", "P2": "25s, PE-20s", "P3": "25s, PE+20s",
         "P4": "25s, uncapped"}
for ts in ("on", "off"):
    conds = [c[0] for c in CONDITIONS if (ts == "on") in [t for t in c[4]] or
             (ts == "off" and False in c[4])]
    conds = [c[0] for c in CONDITIONS if (True if ts == "on" else False) in c[4]]
    print(f"=== timestamps {ts.upper()} ===")
    hdr = f"{'model':>9}{'params':>8}" + "".join(f"{c:>10}" for c in conds) + f"{'recovery P2':>13}"
    print(hdr); print("-" * len(hdr))
    for name in MODELS:
        w = [wer[(name, c, ts)] for c in conds]
        base, late = wer[(name, "P0", ts)], wer[(name, "P1", ts)]
        rec = (late - wer[(name, "P2", ts)]) / (late - base) if abs(late - base) > 1e-9 else float("nan")
        print(f"{name:>9}{ckpt.get(name, {}).get('params_M', 0):>7}M"
              + "".join(f"{v:>10.4f}" for v in w) + f"{rec:>12.2f}x")
    print()

print("recovery of P1's penalty, by condition (timestamps ON):")
print(f"{'model':>9}" + "".join(f"{c:>12}" for c in ("P2", "P3", "P4")))
print("-" * 45)
for name in MODELS:
    base, late = wer[(name, "P0", "on")], wer[(name, "P1", "on")]
    d = late - base
    print(f"{name:>9}" + "".join(
        f"{((late - wer[(name, c, 'on')]) / d if abs(d) > 1e-9 else float('nan')):>12.2f}"
        for c in ("P2", "P3", "P4")))

print("\nrunaway outputs (>3x reference length), timestamps ON:")
print(f"{'model':>9}" + "".join(f"{c:>7}" for c in ("P0","P1","P2","P3","P4")))
for name in MODELS:
    print(f"{name:>9}" + "".join(f"{runaway(name, c, 'on'):>7}" for c in ("P0","P1","P2","P3","P4")))

## 7. Figures

One panel per timestamp arm. Bars are grouped by condition so the within-model comparison — which
is the experiment — reads left to right, and a log y-axis keeps `tiny` (which can exceed 100% WER
at 25 s) on the same plot as `medium` without flattening the others.

In [ ]:
import matplotlib.pyplot as plt

SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e5e5e2"
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
COLOR = {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(MODELS)}


def plot_arm(ts, fname):
    conds = [c[0] for c in CONDITIONS if (True if ts == "on" else False) in c[4]]
    fig, ax = plt.subplots(figsize=(8.4, 4.8), dpi=160)
    fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
    ax.grid(True, axis="y", color=GRID, linewidth=1, which="both")
    ax.set_axisbelow(True)

    x = np.arange(len(conds)); wbar = 0.8 / len(MODELS)
    for k, m in enumerate(MODELS):
        vals = [wer[(m, c, ts)] for c in conds]
        ax.bar(x + (k - (len(MODELS)-1)/2) * wbar, vals, width=wbar * 0.88,
               color=COLOR[m], label=m, edgecolor=SURFACE, linewidth=2, zorder=3)

    ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{c}\n{LABEL[c]}" for c in conds], fontsize=9.5)
    ax.set_ylabel("corpus WER  (log)", fontsize=11, color=INK2)
    ax.set_title(f"Positional embedding vs. utterance position, timestamps {ts.upper()}",
                 fontsize=12.5, color=INK, pad=12, loc="left")
    ax.tick_params(colors=INK2, labelsize=10, length=0)
    for s_ in ("top", "right"):
        ax.spines[s_].set_visible(False)
    for s_ in ("left", "bottom"):
        ax.spines[s_].set_color(GRID); ax.spines[s_].set_linewidth(1)
    leg = ax.legend(frameon=False, fontsize=9.5, ncol=len(MODELS), loc="upper center",
                    bbox_to_anchor=(0.5, -0.22))
    for t_ in leg.get_texts():
        t_.set_color(INK2)
    fig.text(0.005, -0.10,
             f"{len(FILES)} TIMIT TEST utterances - fp16 on CUDA, greedy, batch {BATCH} - "
             f"audio identical in every condition; only the embedding or the decoding rule changes",
             fontsize=7.5, color=INK2)
    fig.tight_layout()
    fig.savefig(fname, bbox_inches="tight", facecolor=SURFACE)
    fig.savefig(os.path.join(DRIVE_ROOT, os.path.basename(fname)),
                bbox_inches="tight", facecolor=SURFACE)
    print(f"wrote {fname} (+ copy on Drive)")


for ts, fn in (("on", "pe_wer_timestamps_on.png"), ("off", "pe_wer_timestamps_off.png")):
    plot_arm(ts, fn)
plt.show()

## 8. Write results and verify

The full record carries hypothesis text and stays on Drive. The per-condition file is corpus-level
numbers only — no paths even — and is safe to commit.

In [ ]:
SAFE_FIELDS = ["model", "params_M", "n_mels", "cond", "offset_s", "pe_shift_s",
               "max_init_ts", "timestamps", "corpus_wer", "runaway", "n_clips"]
FORBIDDEN = {"text", "reference", "hypothesis", "ref", "hyp", "transcript", "path"}
assert not (set(SAFE_FIELDS) & FORBIDDEN)

safe_rows = []
for name in MODELS:
    for cond, off_s, pe_s, mit, arms in CONDITIONS:
        for ts_on in arms:
            ts = "on" if ts_on else "off"
            safe_rows.append({
                "model": name, "params_M": ckpt.get(name, {}).get("params_M", ""),
                "n_mels": ckpt.get(name, {}).get("n_mels", ""), "cond": cond,
                "offset_s": off_s, "pe_shift_s": "" if pe_s is None else pe_s,
                "max_init_ts": "" if mit is None else mit, "timestamps": ts,
                "corpus_wer": f"{wer[(name, cond, ts)]:.6f}",
                "runaway": runaway(name, cond, ts), "n_clips": len(FILES)})
with open(SAFE_CSV, "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=SAFE_FIELDS); wr.writeheader(); wr.writerows(safe_rows)

PROVENANCE["outputs"] = {
    "full_results": {"path": RESULTS_CSV, "contains_text": True, "git_safe": False,
                     "sha256": sha256_file(RESULTS_CSV)},
    "per_condition": {"path": SAFE_CSV, "contains_text": False, "git_safe": True,
                      "sha256": sha256_file(SAFE_CSV)},
}
PROVENANCE["corpus_wer"] = {f"{m}|{c}|{t}": float(v) for (m, c, t), v in wer.items()}
with open(PROV_JSON, "w") as f:
    json.dump(PROVENANCE, f, indent=1)

ok = []
assert len(FILES) == 1000 and len({r["path"] for r in FILES}) == 1000
ok.append("1. corpus: 1000 clips, every sha256_audio verified against the frozen digest")
assert all(v == 0.0 for v in PROVENANCE["mel_gate"]["max_abs_dev_by_n_mels"].values())
ok.append("2. mel gate: 5 s and 25 s bit-identical at n_mels 80 and 128")
exp_rows = sum(len(c[4]) for c in CONDITIONS) * len(FILES) * len(MODELS)
assert len(rows) == exp_rows, f"{len(rows)} rows, expected {exp_rows}"
assert len({(r["model"], r["cond"], r["timestamps"], r["path"]) for r in rows}) == exp_rows
ok.append(f"3. grid: {exp_rows} rows, no duplicates, no gaps")
assert all(v["verified"] for v in ckpt.values()) and len(ckpt) == len(MODELS)
ok.append("4. checkpoints: all verified against the SHA-256 in whisper's download URL")
_pe = whisper.load_model  # buffer restoration is asserted inside rolled_pe on every exit
ok.append("5. positional embedding: restoration asserted on every context exit")
assert len(FILES) % BATCH == 0
ok.append(f"6. batching: every model at batch {BATCH}, {len(FILES)//BATCH} full batches, no remainder")
_back = list(csv.DictReader(open(SAFE_CSV, newline="")))
assert len(_back) == len(safe_rows) and not (set(_back[0]) & FORBIDDEN)
ok.append("7. licensing: git-safe file has no text or path column; hypotheses confined to Drive")
for fn in ("pe_wer_timestamps_on.png", "pe_wer_timestamps_off.png"):
    assert os.path.getsize(fn) > 10000, fn
ok.append("8. figures: both PNGs written")

for line in ok:
    print("PASS  " + line)
print(f"\nper-condition -> {SAFE_CSV}  ({len(safe_rows)} rows, numbers only; safe to commit)")
print(f"full record   -> {RESULTS_CSV}  (contains hypotheses; keep out of git)")
print(f"provenance    -> {PROV_JSON}")